# Transporte entrópico y el algoritmo de Sinkhorn

Este cuaderno acompaña el **Capítulo 13** de las notas del curso (*Transporte entrópico y el algoritmo de Sinkhorn*). Trabajamos con medidas discretas $a\in\Sigma_n$, $b\in\Sigma_m$ (pesos positivos) y una matriz de costos $C$, y estudiamos el problema regularizado

$$
\mathrm{OT}_\varepsilon(a,b)=\min_{\pi\in\Pi(a,b)}\ \langle C,\pi\rangle+\varepsilon\,\mathrm{KL}(\pi\,|\,a\otimes b),
\qquad
\mathrm{KL}(\pi\,|\,a\otimes b)=\sum_{ij}\pi_{ij}\log\frac{\pi_{ij}}{a_ib_j}.
$$

Exploramos computacionalmente:

- La **forma del plan entrópico** $\pi^\varepsilon_{ij}=a_ib_je^{(\varphi^\varepsilon_i+\psi^\varepsilon_j-c_{ij})/\varepsilon}$ y su comparación con el plan del programa lineal.
- El **algoritmo de Sinkhorn** escrito desde cero, en la forma de escalamientos $(u,v)$ y en la forma de potenciales con la $c$-transformada suavizada; la **monotonía del dual** $\mathcal D_\varepsilon$.
- La implementación **estabilizada en el dominio logarítmico**, necesaria para $\varepsilon$ pequeño.
- Los límites $\varepsilon\to0$ (plan óptimo de máxima entropía, potenciales de Kantorovich) y $\varepsilon\to\infty$ (plan producto), con las cotas del capítulo.
- La velocidad de convergencia y su dependencia de $\varepsilon$.
- **Baricentros entrópicos** con el algoritmo de Benamou–Carlier–Cuturi–Nenna–Peyré, recuperando los baricentros explícitos del capítulo anterior, y baricentros de formas en el plano.

Usamos [POT](https://pythonot.github.io/) sólo como control: todo lo esencial está implementado en el notebook en pocas líneas.

In [ ]:
# @title
pip install POT

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
import ot
from scipy.special import logsumexp
from scipy.linalg import sqrtm

rng = np.random.default_rng(7)
np.set_printoptions(precision=4, suppress=True)

## 1. El problema entrópico y la forma del plan

Empezamos con un ejemplo pequeño para poder mirar las matrices: $n=m=6$ puntos en $\mathbb R$ con pesos aleatorios y costo cuadrático. Resolvemos el problema de Kantorovich exacto (LP) y el entrópico para varios $\varepsilon$ (con `ot.sinkhorn`, que implementaremos nosotros en la sección siguiente).

Recordar que POT usa el regularizador $-\varepsilon H(\pi)$, $H(\pi)=-\sum\pi_{ij}(\log\pi_{ij}-1)$, que difiere de $\varepsilon\,\mathrm{KL}(\pi|a\otimes b)$ en una constante sobre $\Pi(a,b)$: el minimizador $\pi^\varepsilon$ es el mismo.

In [ ]:
def KL(pi, a, b):
    ab = np.outer(a, b); m = pi > 0
    return np.sum(pi[m]*np.log(pi[m]/ab[m]))

n = m = 6
x = np.sort(rng.uniform(0, 1, n)); y = np.sort(rng.uniform(0, 1, m))
a = rng.dirichlet(np.ones(n)); b = rng.dirichlet(np.ones(m))
C = (x[:, None] - y[None, :])**2

pi_lp = ot.emd(a, b, C)
print("plan óptimo del LP (un vértice: a lo sumo n+m-1 = 11 entradas no nulas):")
print(pi_lp, "\n  entradas no nulas:", np.sum(pi_lp > 1e-12), "  costo:", np.sum(C*pi_lp))

for eps in [0.1, 0.01, 0.001]:
    pi_eps = ot.sinkhorn(a, b, C, eps, numItermax=100_000, stopThr=1e-12)
    print(f"\nplan entrópico, eps = {eps}:  costo <C,pi> = {np.sum(C*pi_eps):.6f},  KL = {KL(pi_eps, a, b):.4f},  mínimo > 0: {pi_eps.min() > 0}")
    print(pi_eps)

El plan del LP es disperso (un vértice del politopo); el entrópico tiene todas las entradas positivas y, a medida que $\varepsilon$ decrece, concentra la masa donde estaba la del LP y su costo se acerca a $\mathrm{OT}(a,b)$. Verificamos la cota del capítulo

$$
0\le\mathrm{OT}_\varepsilon(a,b)-\mathrm{OT}(a,b)\le\varepsilon\log\frac1{\min_ia_i},
$$

y también la desigualdad **trivial** $\langle C,\pi^\varepsilon\rangle\ge\mathrm{OT}(a,b)$ (el plan entrópico es un plan admisible, así que su costo no puede ser menor que el mínimo).

In [ ]:
OT = np.sum(C*pi_lp)
print(f"{'eps':>8} {'<C,pi_eps>':>12} {'OT_eps':>12} {'OT_eps - OT':>12} {'eps log(1/min a)':>18}")
for eps in [1, 0.3, 0.1, 0.03, 0.01, 0.003]:
    pi_eps = ot.sinkhorn(a, b, C, eps, numItermax=200_000, stopThr=1e-13)
    OTe = np.sum(C*pi_eps) + eps*KL(pi_eps, a, b)
    print(f"{eps:>8} {np.sum(C*pi_eps):>12.6f} {OTe:>12.6f} {OTe-OT:>12.6f} {eps*np.log(1/a.min()):>18.6f}")

## 2. Sinkhorn desde cero

### 2.1 Escalamientos

Con $K^\varepsilon_{ij}=e^{-c_{ij}/\varepsilon}$, el plan entrópico es $\pi^\varepsilon=\operatorname{diag}(u^\varepsilon)K^\varepsilon\operatorname{diag}(v^\varepsilon)$, con $u^\varepsilon_i=a_ie^{\varphi^\varepsilon_i/\varepsilon}$, $v^\varepsilon_j=b_je^{\psi^\varepsilon_j/\varepsilon}$. Imponer las marginales alternadamente da el algoritmo:

$$
u\leftarrow\frac{a}{K^\varepsilon v},\qquad v\leftarrow\frac{b}{(K^\varepsilon)^Tu}.
$$

Cada paso es una normalización de filas o de columnas. Lo implementamos midiendo, en cada iteración, el error en la marginal que **no** se acaba de imponer.

In [ ]:
def sinkhorn_uv(a, b, C, eps, iters=2000, tol=1e-12):
    K = np.exp(-C/eps)
    v = np.ones_like(b); errs = []
    for k in range(iters):
        u = a/(K @ v)                    # ahora las filas suman a
        v = b/(K.T @ u)                  # ahora las columnas suman b (y las filas ya no exactamente)
        err = np.sum(np.abs(u*(K @ v) - a))
        errs.append(err)
        if err < tol: break
    return u, v, K, np.array(errs)

eps = 0.05
u, v, K, errs = sinkhorn_uv(a, b, C, eps)
pi = u[:, None]*K*v[None, :]
print("iteraciones:", len(errs))
print("marginales correctas:", np.allclose(pi.sum(1), a), np.allclose(pi.sum(0), b))
print("coincide con POT:", np.allclose(pi, ot.sinkhorn(a, b, C, eps, numItermax=100_000, stopThr=1e-13), atol=1e-9))

### 2.2 Potenciales y $c$-transformada suavizada

La misma iteración, en los potenciales $\varphi=\varepsilon\log(u/a)$, $\psi=\varepsilon\log(v/b)$, es la alternancia de $c$-transformadas suavizadas:

$$
\varphi\leftarrow(\psi)^{\bar c,\varepsilon},\quad
(\psi)^{\bar c,\varepsilon}_i=-\varepsilon\log\sum_jb_je^{(\psi_j-c_{ij})/\varepsilon};
\qquad
\psi\leftarrow(\varphi)^{c,\varepsilon},\quad
(\varphi)^{c,\varepsilon}_j=-\varepsilon\log\sum_ia_ie^{(\varphi_i-c_{ij})/\varepsilon}.
$$

Cada paso maximiza exactamente el dual entrópico

$$
\mathcal D_\varepsilon(\varphi,\psi)=\langle a,\varphi\rangle+\langle b,\psi\rangle-\varepsilon\sum_{ij}a_ib_je^{(\varphi_i+\psi_j-c_{ij})/\varepsilon}+\varepsilon
$$

en una de las dos variables, de modo que $\mathcal D_\varepsilon$ **crece en cada paso** y converge a $\mathrm{OT}_\varepsilon(a,b)$. Implementamos esta versión usando `logsumexp`, que es numéricamente estable: es la implementación **en el dominio logarítmico**, y es la que se usa en la práctica.

In [ ]:
def ctransf_eps(phi, a, C, eps):
    """(phi)^{c,eps}_j = -eps log sum_i a_i exp((phi_i - c_ij)/eps), estable."""
    return -eps*logsumexp((phi[:, None] - C)/eps, b=a[:, None], axis=0)

def cbar_transf_eps(psi, b, C, eps):
    return -eps*logsumexp((psi[None, :] - C)/eps, b=b[None, :], axis=1)

def dual_eps(phi, psi, a, b, C, eps):
    return a @ phi + b @ psi - eps*np.sum(np.outer(a, b)*np.exp((phi[:, None] + psi[None, :] - C)/eps)) + eps

def sinkhorn_log(a, b, C, eps, iters=5000, tol=1e-12):
    psi = np.zeros_like(b); D = []
    for k in range(iters):
        phi = cbar_transf_eps(psi, b, C, eps)
        psi_new = ctransf_eps(phi, a, C, eps)
        D.append(dual_eps(phi, psi_new, a, b, C, eps))
        if np.max(np.abs(psi_new - psi)) < tol: psi = psi_new; break
        psi = psi_new
    pi = np.outer(a, b)*np.exp((phi[:, None] + psi[None, :] - C)/eps)
    return phi, psi, pi, np.array(D)

phi, psi, pi_log, D = sinkhorn_log(a, b, C, eps)
OTe = np.sum(C*pi_log) + eps*KL(pi_log, a, b)
print("D_eps es no decreciente:", np.all(np.diff(D) >= -1e-13))
print("D_eps final =", D[-1], "   OT_eps (primal) =", OTe, "   (dualidad fuerte)")
print("mismo plan que la versión (u,v):", np.allclose(pi_log, pi, atol=1e-10))
print("relaciones psi = (phi)^{c,eps}, phi = (psi)^{cbar,eps}:",
      np.allclose(psi, ctransf_eps(phi, a, C, eps)), np.allclose(phi, cbar_transf_eps(psi, b, C, eps)))

plt.figure(figsize=(6, 3.5)); plt.semilogy(OTe - D + 1e-17, 'o-', ms=3)
plt.xlabel('iteración'); plt.ylabel(r'$\mathrm{OT}_\varepsilon-\mathcal{D}_\varepsilon(\varphi^{(k)},\psi^{(k)})$'); plt.title('Maximización alternada del dual'); plt.show()

### 2.3 Por qué hace falta el dominio logarítmico

Con $\varepsilon$ pequeño, $K^\varepsilon_{ij}=e^{-c_{ij}/\varepsilon}$ tiene entradas del orden de $e^{-1/\varepsilon}$: para $\varepsilon=0.001$ y costos de orden $1$, eso es $10^{-434}$, que en doble precisión (cuyo menor número positivo es del orden de $10^{-308}$) es exactamente cero. La versión $(u,v)$ divide entonces por cero, mientras que la versión logarítmica sólo maneja números del orden de los costos.

In [ ]:
eps_chico = 0.001
C2 = (x[:, None] - (y[None, :] + 2))**2          # el mismo problema con nu trasladada: todos los costos son >= 1
OT2 = ot.emd2(a, b, C2)
with np.errstate(all='ignore'):
    u_, v_, K_, errs_ = sinkhorn_uv(a, b, C2, eps_chico, iters=50)
    pi_uv = u_[:, None]*K_*v_[None, :]
print("versión (u,v): entradas de K^eps no nulas:", np.sum(K_ > 0), "de", K_.size, "; el plan contiene NaN:", np.isnan(pi_uv).any())

phi_, psi_, pi_l, D_ = sinkhorn_log(a, b, C2, eps_chico, iters=20000, tol=1e-10)
print("versión log-domain: marginales correctas:", np.allclose(pi_l.sum(1), a, atol=1e-8), np.allclose(pi_l.sum(0), b, atol=1e-8),
      "; iteraciones:", len(D_), "; costo:", np.sum(C2*pi_l), " vs OT =", OT2)

## 3. Velocidad de convergencia

La demostración elemental del capítulo da convergencia sin tasa; la demostración de Franklin–Lorenz da convergencia lineal en la métrica de Hilbert, con una razón que se deteriora cuando $\varepsilon\to0$ (la razón se acerca a $1$ como $1-e^{-\Delta/\varepsilon}$ aproximadamente, con $\Delta$ la oscilación del costo). Lo vemos midiendo el error en la marginal a lo largo de las iteraciones, para varios $\varepsilon$, en un problema algo más grande.

In [ ]:
n = m = 200
X = rng.uniform(0, 1, (n, 2)); Y = rng.uniform(0, 1, (m, 2)) + 0.3
a = np.ones(n)/n; b = np.ones(m)/m
C = ot.dist(X, Y)          # euclídea al cuadrado

def sinkhorn_log_errs(a, b, C, eps, iters):
    psi = np.zeros_like(b); errs = []
    for k in range(iters):
        phi = cbar_transf_eps(psi, b, C, eps)
        psi = ctransf_eps(phi, a, C, eps)
        pi = np.outer(a, b)*np.exp((phi[:, None] + psi[None, :] - C)/eps)
        errs.append(np.sum(np.abs(pi.sum(1) - a)))
    return np.array(errs)

plt.figure(figsize=(7, 4))
for eps in [0.3, 0.1, 0.03, 0.01, 0.003]:
    e = sinkhorn_log_errs(a, b, C, eps, 400)
    plt.semilogy(e, label=fr'$\varepsilon={eps}$')
plt.xlabel('iteración'); plt.ylabel(r'$\|\pi^{(k)}\mathbf{1}-a\|_1$'); plt.legend()
plt.title(r'Convergencia lineal, más lenta cuanto menor es $\varepsilon$'); plt.show()

Para $\varepsilon$ moderado, el algoritmo converge en decenas de iteraciones; para $\varepsilon$ pequeño se necesitan miles. Como cada iteración cuesta $O(nm)$, el costo total para $\varepsilon$ moderado es incomparablemente menor que el del simplex para $n$ grande, y además las operaciones son productos matriz-vector, que se paralelizan trivialmente: esa es la razón del éxito práctico del método.

## 4. Los límites $\varepsilon\to\infty$ y $\varepsilon\to0$

### 4.1 $\varepsilon\to\infty$: el plan producto

Cuando $\varepsilon\to\infty$ el término entrópico domina y $\pi^\varepsilon\to a\otimes b$, el único minimizador de $\mathrm{KL}(\cdot\,|\,a\otimes b)$. En el ejemplo de $6\times6$:

In [ ]:
n = m = 6
x = np.sort(rng.uniform(0, 1, n)); y = np.sort(rng.uniform(0, 1, m))
a = rng.dirichlet(np.ones(n)); b = rng.dirichlet(np.ones(m))
C = (x[:, None] - y[None, :])**2

print(f"{'eps':>8} {'||pi_eps - a x b||_1':>22}")
for eps in [0.01, 0.1, 1, 10, 100, 1000]:
    _, _, pi_eps, _ = sinkhorn_log(a, b, C, eps)
    print(f"{eps:>8} {np.sum(np.abs(pi_eps - np.outer(a, b))):>22.2e}")

### 4.2 $\varepsilon\to0$: el plan óptimo de máxima entropía

Cuando $\varepsilon\to0$, $\pi^\varepsilon$ converge al plan óptimo del problema de Kantorovich que **minimiza $\mathrm{KL}(\cdot\,|\,a\otimes b)$ entre todos los planes óptimos**. Si el plan óptimo es único, simplemente converge a él. El caso interesante es el degenerado: elegimos un ejemplo con muchos planes óptimos, $a=b$ uniformes sobre $\{0,1,2,3\}$ y $\{1,2,3,4\}$ con costo $|x-y|$, donde $W_1=1$ se alcanza en muchos planes (mover cada punto un lugar, o mover sólo el $0$ hasta el $4$, o…).

In [ ]:
x = np.arange(4.); y = np.arange(1., 5.)
a = b = np.ones(4)/4
C = np.abs(x[:, None] - y[None, :])
OT = ot.emd2(a, b, C)
pi_vertice = ot.emd(a, b, C)
print("W_1 =", OT, "\nun plan óptimo (vértice) que devuelve el LP:\n", pi_vertice, "\n  KL =", KL(pi_vertice, a, b))

# otros dos planes óptimos escritos a mano
pi_shift = np.eye(4)/4                                   # i -> i+1 (x_i -> y_i)
pi_salto = np.zeros((4, 4)); pi_salto[0, 3] = 1/4; pi_salto[1, 0] = pi_salto[2, 1] = pi_salto[3, 2] = 1/4   # 0->4, y los demás quietos
for nombre, p in [("desplazar todos un lugar", pi_shift), ("saltar 0 -> 4", pi_salto)]:
    print(f"{nombre}: costo = {np.sum(C*p):.4f}, KL = {KL(p, a, b):.4f}")

print("\nlímite eps -> 0 del plan entrópico:")
anterior = None
for eps in [0.3, 0.1, 0.03, 0.01, 0.003, 0.001]:
    _, _, pi_eps, _ = sinkhorn_log(a, b, C, eps, iters=50000, tol=1e-13)
    cambio = "" if anterior is None else f"  ||pi_eps - pi_anterior||_1 = {np.sum(np.abs(pi_eps - anterior)):.2e}"
    print(f"eps = {eps:<6}  costo = {np.sum(C*pi_eps):.6f}  KL = {KL(pi_eps, a, b):.5f}{cambio}")
    anterior = pi_eps
print("\npi^0 (numérico):\n", pi_eps)

El límite $\pi^0$ es óptimo (su costo es $W_1=1$), tiene entropía relativa **menor** que la de los vértices (es una combinación convexa de varios de ellos), y no coincide con ninguno de los planes "naturales". Es el plan óptimo que Sinkhorn selecciona entre todos los posibles: la regularización actúa como criterio de selección.

### 4.3 Los potenciales entrópicos convergen a potenciales de Kantorovich

En un problema **no degenerado**, los potenciales $(\varphi^\varepsilon,\psi^\varepsilon)$ convergen, salvo la constante aditiva, a los potenciales de Kantorovich del LP. Lo verificamos con puntos aleatorios (el LP es entonces no degenerado con probabilidad 1) comparando con los potenciales duales que devuelve `ot.emd(..., log=True)`, normalizando $\varphi_1=0$.

In [ ]:
n = m = 8
x = rng.uniform(0, 1, n); y = rng.uniform(0, 1, m)
a = rng.dirichlet(np.ones(n)); b = rng.dirichlet(np.ones(m))
C = (x[:, None] - y[None, :])**2

_, log = ot.emd(a, b, C, log=True)
phi_K, psi_K = log['u'] - log['u'][0], log['v'] + log['u'][0]
print("potenciales de Kantorovich (LP), normalizados phi_1 = 0:\n", phi_K, "\n", psi_K)
print("valor dual =", a @ phi_K + b @ psi_K, "= OT =", ot.emd2(a, b, C))

print(f"\n{'eps':>8} {'||phi_eps - phi_K||_inf':>24} {'||psi_eps - psi_K||_inf':>24} {'D_eps':>10}")
for eps in [0.1, 0.03, 0.01, 0.003, 0.001]:
    phi_e, psi_e, pi_e, D = sinkhorn_log(a, b, C, eps, iters=100000, tol=1e-13)
    s = phi_e[0]; phi_e, psi_e = phi_e - s, psi_e + s
    print(f"{eps:>8} {np.max(np.abs(phi_e - phi_K)):>24.2e} {np.max(np.abs(psi_e - psi_K)):>24.2e} {D[-1]:>10.6f}")

Los potenciales entrópicos convergen a los del LP cuando $\varepsilon\to0$, y el valor dual $\mathcal D_\varepsilon\to\mathrm{OT}$. Esto es lo que dice la observación del capítulo: el límite $\varepsilon\to0$ de los potenciales entrópicos da una **construcción alternativa de los potenciales de Kantorovich discretos**, sin pasar por el teorema de dualidad de la programación lineal.

**Ejercicio.** Repetir con $a=b$ uniformes y $x,y$ equiespaciados (problema degenerado). ¿Convergen los potenciales? ¿A qué? Comparar con la no unicidad de los potenciales de Kantorovich en ese caso.

## 5. Baricentros entrópicos

Sobre una grilla fija $\{x_1,\dots,x_n\}$ y con medidas $b^1,\dots,b^N$ sobre ella, el **baricentro entrópico** es el minimizador en $a\in\Sigma_n$ de $\mathcal F_\varepsilon(a)=\sum_k\lambda_k\,\mathrm{OT}_\varepsilon(a,b^k)$. El algoritmo de Benamou, Carlier, Cuturi, Nenna y Peyré escribe cada plan como $\pi^k=\operatorname{diag}(u^k)K^\varepsilon\operatorname{diag}(v^k)$ y alterna:

1. $u^k\leftarrow a/(K^\varepsilon v^k)$ para cada $k$;
2. $v^k\leftarrow b^k/((K^\varepsilon)^Tu^k)$ para cada $k$;
3. $a\leftarrow\prod_k\bigl(u^k\odot K^\varepsilon v^k\bigr)^{\lambda_k}$ (media geométrica ponderada de las primeras marginales).

Lo implementamos tal cual, en la versión $(u,v)$ y en la versión estabilizada en el dominio logarítmico (la primera falla en cuanto $K^\varepsilon$ tiene ceros numéricos), y lo usamos para **recuperar numéricamente los baricentros explícitos del capítulo anterior**.

In [ ]:
def baricentro_sinkhorn(B, C, lam, eps, iters=2000, tol=1e-10):
    """B: matriz n x N con las medidas b^k en columnas. Devuelve el baricentro a en la misma grilla."""
    n, N = B.shape
    K = np.exp(-C/eps)
    V = np.ones((n, N)); a = np.ones(n)/n
    for it in range(iters):
        U = a[:, None]/(K @ V)
        V = B/(K.T @ U)
        a_new = np.prod((U*(K @ V))**lam[None, :], axis=1)
        a_new /= a_new.sum()                       # (la media geométrica no tiene masa 1 exactamente hasta converger)
        if np.sum(np.abs(a_new - a)) < tol: a = a_new; break
        a = a_new
    return a

def baricentro_sinkhorn_log(B, C, lam, eps, iters=2000, tol=1e-10):
    """El mismo algoritmo en el dominio logarítmico: f^k = eps log u^k, g^k = eps log v^k."""
    n, N = B.shape
    logB = np.log(B + 1e-300)
    Gm = np.zeros((n, N)); loga = np.full(n, -np.log(n))
    for it in range(iters):
        # paso 1: u^k = a / (K v^k)          ->  f^k = eps log a - eps LSE_j((g^k_j - c_ij)/eps)
        Fm = np.stack([eps*loga - eps*logsumexp((Gm[None, :, k] - C)/eps, axis=1) for k in range(N)], axis=1)
        # paso 2: v^k = b^k / (K^T u^k)      ->  g^k = eps log b^k - eps LSE_i((f^k_i - c_ij)/eps)
        Gm = np.stack([eps*logB[:, k] - eps*logsumexp((Fm[:, None, k] - C)/eps, axis=0) for k in range(N)], axis=1)
        # paso 3: log(u^k (K v^k))_i = f^k_i/eps + LSE_j((g^k_j - c_ij)/eps);  log a = sum_k lam_k (...)
        logmarg = np.stack([Fm[:, k]/eps + logsumexp((Gm[None, :, k] - C)/eps, axis=1) for k in range(N)], axis=1)
        loga_new = logmarg @ lam; loga_new -= logsumexp(loga_new)
        if np.sum(np.abs(np.exp(loga_new) - np.exp(loga))) < tol: loga = loga_new; break
        loga = loga_new
    return np.exp(loga)

### 5.1 Dimensión uno: el promedio de cuantiles

Tomamos tres medidas en una grilla de $[-5,8]$ y comparamos el baricentro entrópico, para $\varepsilon$ decreciente, con el promedio de cuantiles $F^{[-1]}=\sum\lambda_kF_k^{[-1]}$, que es el baricentro exacto.

In [ ]:
grilla = np.linspace(-5, 8, 301); h = grilla[1] - grilla[0]
C = (grilla[:, None] - grilla[None, :])**2
lam = np.array([0.4, 0.4, 0.2])

def densidad_en_grilla(f):
    p = f(grilla); return p/p.sum()
from scipy.stats import norm, expon
B = np.stack([densidad_en_grilla(lambda t: norm.pdf(t, -3, 0.6)),
              densidad_en_grilla(lambda t: 0.5*norm.pdf(t, 0, 0.3) + 0.5*norm.pdf(t, 1.5, 0.3)),
              densidad_en_grilla(lambda t: expon.pdf(t - 3, scale=1.0))], axis=1)

# baricentro exacto por cuantiles, con las pseudoinversas de las distribuciones continuas
ts = (np.arange(400_000) + 0.5)/400_000
tt = np.linspace(-6, 9, 200_001)
F_mix = 0.5*norm.cdf(tt, 0, 0.3) + 0.5*norm.cdf(tt, 1.5, 0.3)
q_mix = np.interp(ts, F_mix, tt)                          # inversión numérica de la FDA de la mezcla
q_bar = lam[0]*norm.ppf(ts, -3, 0.6) + lam[1]*q_mix + lam[2]*(expon.ppf(ts, scale=1.0) + 3)
a_exacto, _ = np.histogram(q_bar, bins=np.concatenate([grilla - h/2, [grilla[-1] + h/2]])); a_exacto = a_exacto/a_exacto.sum()

fig, ax = plt.subplots(figsize=(10, 4))
for k in range(3): ax.fill_between(grilla, B[:, k]/h, alpha=0.3, label=fr'$b^{k+1}$')
ax.plot(grilla, a_exacto/h, 'k', lw=2.5, label='baricentro exacto (cuantiles)')
for eps, ls in [(0.5, ':'), (0.1, '--'), (0.02, '-')]:
    a_eps = baricentro_sinkhorn_log(B, C, lam, eps, iters=5000)
    ax.plot(grilla, a_eps/h, ls, lw=1.8, label=fr'entrópico, $\varepsilon={eps}$')
    print(f"eps = {eps}: W_2(baricentro entrópico, exacto) = {np.sqrt(ot.emd2(a_eps, a_exacto, C)):.4f}")
ax.legend(); ax.set_title('Baricentro entrópico vs. promedio de cuantiles'); plt.show()

Para $\varepsilon$ grande el baricentro entrópico está **suavizado** (la entropía penaliza las concentraciones); cuando $\varepsilon\to0$ converge al baricentro exacto. Este sesgo de suavizado es el precio de la regularización, y en la práctica se elige $\varepsilon$ del orden del cuadrado del paso de la grilla.

### 5.2 Gaussianas en el plano: el punto fijo

Discretizamos tres gaussianas en una grilla de $61\times61$ puntos de $[-3,6]^2$ y comparamos la media y la covarianza del baricentro entrópico con la solución $\bar\Sigma$ de la ecuación de punto fijo $\bar\Sigma=\sum\lambda_k(\bar\Sigma^{1/2}\Sigma_k\bar\Sigma^{1/2})^{1/2}$.

In [ ]:
g = np.linspace(-3, 6, 61); GX, GY = np.meshgrid(g, g, indexing='ij'); G = np.c_[GX.ravel(), GY.ravel()]
C = ot.dist(G, G)

def rot(th): return np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
Sigmas = [rot(th) @ np.diag([2.0, 0.25]) @ rot(th).T for th in [0, np.pi/3, 2*np.pi/3]]
medias = [np.array([0., 0.]), np.array([3., 0.]), np.array([1.5, 2.5])]
lam = np.ones(3)/3

def gauss_en_grilla(m, S):
    d = G - m; p = np.exp(-0.5*np.einsum('ij,jk,ik->i', d, np.linalg.inv(S), d)); return p/p.sum()
B = np.stack([gauss_en_grilla(m, S) for m, S in zip(medias, Sigmas)], axis=1)

# punto fijo exacto
Sbar = np.mean(Sigmas, axis=0)
for _ in range(200):
    Sh = np.real(sqrtm(Sbar)); Sbar = sum(l*np.real(sqrtm(Sh @ S @ Sh)) for l, S in zip(lam, Sigmas))
mbar = sum(l*m for l, m in zip(lam, medias))

eps = 0.05
a_eps = baricentro_sinkhorn(B, C, lam, eps, iters=3000)
m_emp = a_eps @ G; S_emp = (G - m_emp).T @ ((G - m_emp)*a_eps[:, None])
print("media exacta      :", mbar, "\nmedia entrópica   :", m_emp)
print("covarianza exacta :\n", Sbar, "\ncovarianza entrópica (eps = %.2f):\n" % eps, S_emp)
print("paso de la grilla h =", round(g[1]-g[0], 3))

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for k in range(3):
    ax[k].contourf(GX, GY, B[:, k].reshape(GX.shape), levels=12, cmap='Blues'); ax[k].set_title(fr'$b^{k+1}$')
ax[3].contourf(GX, GY, a_eps.reshape(GX.shape), levels=12, cmap='Reds'); ax[3].set_title(fr'baricentro entrópico, $\varepsilon={eps}$')
th = np.linspace(0, 2*np.pi, 200); w, V = np.linalg.eigh(Sbar); el = (V*np.sqrt(w)) @ np.vstack([np.cos(th), np.sin(th)])*2
ax[3].plot(mbar[0] + el[0], mbar[1] + el[1], 'k--', lw=1.5, label='elipse exacta (2 desvíos)'); ax[3].legend(fontsize=8)
for a_ in ax: a_.set_aspect('equal')
plt.show()

El baricentro entrópico recupera la media y la covarianza del punto fijo salvo errores del orden del paso de la grilla, del truncamiento de las colas al cuadrado $[-3,6]^2$ y del sesgo entrópico, y la elipse exacta se superpone a las curvas de nivel del baricentro calculado.

### 5.3 Baricentros de formas

Cerramos con el ejemplo que hizo populares a los baricentros de Wasserstein: interpolar entre **formas** vistas como medidas uniformes en el plano. Con tres formas y pesos $(\lambda_1,\lambda_2,\lambda_3)$ variando en el triángulo de pesos, se obtiene una familia de formas intermedias. Aquí usamos la implementación de POT `ot.bregman.convolutional_barycenter2d` (Solomon et al., 2015): es el mismo algoritmo de la Sección 5, pero en una grilla regular con costo cuadrático el núcleo $K^\varepsilon$ es una gaussiana separable, y multiplicar por $K^\varepsilon$ se reduce a dos convoluciones unidimensionales de costo $O(N^3)$ en vez de $O(N^4)$ para imágenes de $N\times N$. Eso permite grillas más finas y $\varepsilon$ más chico. (El costo en POT está normalizado a la grilla $[0,1]^2$.)

In [ ]:
N = 50
g = np.linspace(-1, 1, N); GX, GY = np.meshgrid(g, g, indexing='ij'); G = np.c_[GX.ravel(), GY.ravel()]
disco = (GX**2 + GY**2 <= 0.5**2).astype(float)
cuadrado = ((np.abs(GX) <= 0.45) & (np.abs(GY) <= 0.45)).astype(float)
triangulo = ((GY >= -0.5) & (GY <= 0.5 - np.sqrt(3)*np.abs(GX)) ).astype(float)
formas = [disco, cuadrado, triangulo]
A = np.stack([f/f.sum() for f in formas])            # imágenes N x N normalizadas
eps = 0.001

fig, ax = plt.subplots(5, 5, figsize=(10, 10))
for i in range(5):
    for j in range(5):
        ax[i, j].axis('off')
        if i + j > 4: continue
        # coordenadas baricéntricas en el triángulo de pesos
        l1, l2 = (4 - i - j)/4, j/4; l3 = 1 - l1 - l2
        w = np.array([l1, l2, l3])
        bar = ot.bregman.convolutional_barycenter2d(A, eps, weights=w, numItermax=3000, stopThr=1e-8)
        ax[i, j].imshow(bar.T, origin='lower', cmap='Greys')
        ax[i, j].set_title(f'({l1:.2f}, {l2:.2f}, {l3:.2f})', fontsize=8)
plt.suptitle(r'Baricentros entrópicos de tres formas; pesos $(\lambda_{\rm disco},\lambda_{\rm cuadrado},\lambda_{\rm triángulo})$')
plt.tight_layout(); plt.show()

Las formas intermedias (levemente difuminadas por la entropía, incluso en los vértices del triángulo) son genuinas interpolaciones geométricas —un disco que se va cuadrando, un cuadrado que se va afinando hacia un triángulo—, no superposiciones de las tres formas originales, que es lo que daría la mezcla $\sum\lambda_kb^k$. Esto es exactamente la diferencia entre la geometría de $W_2$ y la geometría lineal de las medidas que recorrió todo el curso.

## Ejercicios computacionales

Los enunciados siguientes figuran también en la sección de ejercicios del capítulo correspondiente de las notas.

1. **Convergencia de valores.** Graficar $\varepsilon\mapsto\mathrm{OT}_\varepsilon(a,b)-\mathrm{OT}(a,b)$ en escala log-log para el ejemplo de la Sección 1 y estimar el orden de convergencia. ¿Es $O(\varepsilon)$ como dice la cota, o mejor? (Para el problema no degenerado, la convergencia real es $O(\varepsilon\log(1/\varepsilon))$ o incluso $O(\varepsilon)$ con una constante menor que $\log(1/\min a_i)$.)

2. **Sinkhorn como iteración $\varphi\mapsto\varphi^{c\bar c}$.** Implementar la mejora de potenciales del capítulo de dualidad, $\varphi\mapsto(\varphi^c)^{\bar c}$, sin regularización, y compararla con la iteración de Sinkhorn para $\varepsilon$ pequeño partiendo del mismo $\varphi$. ¿Converge la versión no regularizada? (No en general: puede oscilar; ésa es una de las razones para regularizar.)

3. **Coste de cada iteración.** Medir el tiempo de una iteración de Sinkhorn para $n=m=500,1000,2000,4000$ y verificar el crecimiento $O(nm)$. Comparar con el tiempo de `ot.emd` para los mismos tamaños.

4. **Un baricentro con $\varepsilon\to0$.** En la Sección 5.1, calcular el baricentro entrópico para $\varepsilon\in\{0.5,0.2,0.1,0.05,0.02,0.01,0.005\}$ y graficar $W_2$(baricentro entrópico, exacto) en función de $\varepsilon$, en escala log-log. ¿Cuántas iteraciones hacen falta en cada caso?

5. **Unicidad del baricentro entrópico.** Verificar numéricamente que el baricentro entrópico del ejemplo de no unicidad del capítulo anterior ($\mu_1$ con átomos en $(\pm1,0)$, $\mu_2$ con átomos en $(0,\pm1)$, sobre una grilla que contenga los cuatro candidatos) es **único** y simétrico: la entropía rompe la degeneración eligiendo la combinación más "repartida".